<a href="https://colab.research.google.com/github/mhabibier/Practical-Statistics-for-Data-Scientists/blob/main/01_Exploratory_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1 — Exploratory Data Analysis

**Book:** Practical Statistics for Data Scientists, Second Edition  
**Authors:** Peter Bruce, Andrew Bruce, and Peter Gedeck  
**Student:** Muhammad Habibie Rabbani  
**Student ID:** [Isi NIM]  
**Study Program:** Computer Engineering  
**University:** Telkom University  

Notebook ini berisi reproduksi kode, penjelasan teori, dan analisis
Chapter 1 mengenai Exploratory Data Analysis.

## Tujuan Pembelajaran

Setelah menyelesaikan chapter ini, saya diharapkan mampu:

1. Memahami tipe-tipe data dalam data science.
2. Menghitung ukuran pemusatan data.
3. Menghitung ukuran penyebaran data.
4. Menjelaskan distribusi menggunakan persentil dan visualisasi.
5. Mengeksplorasi data kategorikal.
6. Mengukur hubungan antara dua variabel menggunakan korelasi.
7. Memvisualisasikan hubungan dua atau lebih variabel.

## 1. Exploratory Data Analysis

Exploratory Data Analysis atau EDA adalah proses memeriksa,
meringkas, dan memvisualisasikan data sebelum membangun model statistik
atau machine learning.

EDA digunakan untuk:

- Memahami struktur dataset.
- Mengetahui tipe setiap variabel.
- Menemukan nilai yang hilang.
- Menemukan outlier.
- Melihat distribusi data.
- Menemukan pola dan hubungan antarvariabel.
- Membantu menentukan metode analisis yang tepat.

### Tipe Data

Data numerik terdiri dari:

1. **Continuous data**, yaitu data yang dapat memiliki nilai pada suatu
   rentang, misalnya tinggi badan dan suhu.
2. **Discrete data**, yaitu data hasil perhitungan, misalnya jumlah
   kendaraan dan jumlah mahasiswa.

Data kategorikal terdiri dari:

1. **Nominal data**, yaitu kategori tanpa urutan, misalnya kota atau warna.
2. **Ordinal data**, yaitu kategori yang mempunyai urutan, misalnya
   tingkat kepuasan.
3. **Binary data**, yaitu data dengan dua kemungkinan nilai, misalnya
   ya/tidak atau berhasil/gagal.

In [ ]:
%pip -q install wquantiles statsmodels

In [ ]:
from pathlib import Path
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import trim_mean
from statsmodels import robust
import wquantiles

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)

sns.set_theme(
    style="whitegrid",
    palette="deep"
)

print("Semua library berhasil diimpor.")

In [ ]:
REPOSITORY_URL = (
    "https://github.com/gedeck/"
    "practical-statistics-for-data-scientists.git"
)

REPOSITORY_DIRECTORY = Path(
    "/content/practical-statistics-for-data-scientists"
)

DATA_DIRECTORY = REPOSITORY_DIRECTORY / "data"

if not REPOSITORY_DIRECTORY.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            REPOSITORY_URL,
            str(REPOSITORY_DIRECTORY)
        ],
        check=True
    )

if not DATA_DIRECTORY.exists():
    raise FileNotFoundError(
        "Folder dataset tidak berhasil ditemukan."
    )

print("Lokasi dataset:", DATA_DIRECTORY)

In [ ]:
state = pd.read_csv(
    DATA_DIRECTORY / "state.csv"
)

dfw_airline = pd.read_csv(
    DATA_DIRECTORY / "dfw_airline.csv"
)

sp500_data = pd.read_csv(
    DATA_DIRECTORY / "sp500_data.csv.gz",
    index_col=0
)

sp500_data.index = pd.to_datetime(
    sp500_data.index
)

sp500_sectors = pd.read_csv(
    DATA_DIRECTORY / "sp500_sectors.csv"
)

kc_tax = pd.read_csv(
    DATA_DIRECTORY / "kc_tax.csv.gz"
)

lc_loans = pd.read_csv(
    DATA_DIRECTORY / "lc_loans.csv"
)

airline_stats = pd.read_csv(
    DATA_DIRECTORY / "airline_stats.csv"
)

dataset_summary = pd.DataFrame({
    "Dataset": [
        "state",
        "dfw_airline",
        "sp500_data",
        "sp500_sectors",
        "kc_tax",
        "lc_loans",
        "airline_stats"
    ],
    "Rows": [
        state.shape[0],
        dfw_airline.shape[0],
        sp500_data.shape[0],
        sp500_sectors.shape[0],
        kc_tax.shape[0],
        lc_loans.shape[0],
        airline_stats.shape[0]
    ],
    "Columns": [
        state.shape[1],
        dfw_airline.shape[1],
        sp500_data.shape[1],
        sp500_sectors.shape[1],
        kc_tax.shape[1],
        lc_loans.shape[1],
        airline_stats.shape[1]
    ]
})

dataset_summary

In [ ]:
display(state.head())

print("\nInformasi dataset:")
state.info()

print("\nJumlah missing value:")
display(state.isna().sum())

### Analisis Awal Dataset

Dataset `state` memiliki 50 baris yang merepresentasikan negara bagian
di Amerika Serikat.

Variabel yang digunakan adalah:

- `State`: nama negara bagian.
- `Population`: jumlah penduduk.
- `Murder.Rate`: tingkat pembunuhan per 100.000 penduduk.
- `Abbreviation`: singkatan negara bagian.

`Population` dan `Murder.Rate` merupakan data numerik, sedangkan
`State` dan `Abbreviation` merupakan data kategorikal nominal.

## 2. Estimates of Location

Estimates of location digunakan untuk menggambarkan posisi pusat dari
sekumpulan data.

### Mean

Mean atau rata-rata diperoleh dengan menjumlahkan seluruh nilai dan
membaginya dengan jumlah data.

$$
\bar{x} = \frac{\sum_{i=1}^{n}x_i}{n}
$$

Mean mudah dipahami, tetapi sensitif terhadap outlier.

### Median

Median adalah nilai tengah setelah data diurutkan. Median lebih tahan
terhadap outlier dibandingkan mean.

### Trimmed Mean

Trimmed mean menghitung rata-rata setelah membuang sejumlah nilai
terkecil dan terbesar. Metode ini mengurangi pengaruh nilai ekstrem.

### Weighted Mean

Weighted mean memberikan bobot yang berbeda pada setiap data.

$$
\bar{x}_w = \frac{\sum w_ix_i}{\sum w_i}
$$

### Weighted Median

Weighted median adalah nilai ketika jumlah bobot kumulatif telah
mencapai setengah dari seluruh bobot.

In [ ]:
population_location = pd.DataFrame({
    "Measure": [
        "Mean",
        "Trimmed Mean 10%",
        "Median"
    ],
    "Value": [
        state["Population"].mean(),
        trim_mean(
            state["Population"],
            proportiontocut=0.10
        ),
        state["Population"].median()
    ]
})

population_location.style.format({
    "Value": "{:,.2f}"
})

In [ ]:
murder_rate_location = pd.DataFrame({
    "Measure": [
        "Mean",
        "Population-Weighted Mean",
        "Population-Weighted Median"
    ],
    "Value": [
        state["Murder.Rate"].mean(),
        np.average(
            state["Murder.Rate"],
            weights=state["Population"]
        ),
        wquantiles.median(
            state["Murder.Rate"],
            weights=state["Population"]
        )
    ]
})

murder_rate_location.style.format({
    "Value": "{:.3f}"
})

### Analisis Estimates of Location

Rata-rata populasi adalah sekitar 6,16 juta, sedangkan mediannya sekitar
4,44 juta. Mean yang lebih besar daripada median menunjukkan bahwa
distribusi populasi condong ke kanan. Beberapa negara bagian dengan
populasi sangat besar menaikkan nilai rata-rata.

Trimmed mean sebesar sekitar 4,78 juta berada di antara mean dan median.
Hal ini menunjukkan bahwa trimmed mean berhasil mengurangi pengaruh
negara bagian dengan populasi ekstrem.

Mean tingkat pembunuhan tanpa bobot adalah sekitar 4,066. Setelah
populasi digunakan sebagai bobot, weighted mean menjadi sekitar 4,446.
Artinya, negara bagian dengan populasi besar memberikan pengaruh lebih
besar pada hasil perhitungan berbobot.

## 3. Estimates of Variability

Ukuran pemusatan belum cukup untuk menjelaskan data. Dua dataset dapat
mempunyai mean yang sama, tetapi tingkat penyebaran yang berbeda.

### Range

Range adalah selisih nilai maksimum dan minimum.

$$
Range = x_{max} - x_{min}
$$

### Variance

Variance menghitung rata-rata kuadrat selisih data terhadap mean.

$$
s^2 = \frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}
$$

### Standard Deviation

Standard deviation adalah akar dari variance.

$$
s = \sqrt{s^2}
$$

### Interquartile Range

Interquartile range atau IQR adalah selisih persentil ke-75 dan
persentil ke-25.

$$
IQR = Q_3-Q_1
$$

### Median Absolute Deviation

MAD mengukur median jarak absolut setiap data terhadap median.
MAD lebih tahan terhadap outlier dibandingkan standard deviation.

In [ ]:
population_variability = pd.DataFrame({
    "Measure": [
        "Range",
        "Variance",
        "Standard Deviation",
        "Interquartile Range",
        "Normalized MAD"
    ],
    "Value": [
        (
            state["Population"].max()
            - state["Population"].min()
        ),
        state["Population"].var(),
        state["Population"].std(),
        (
            state["Population"].quantile(0.75)
            - state["Population"].quantile(0.25)
        ),
        robust.scale.mad(
            state["Population"]
        )
    ]
})

population_variability.style.format({
    "Value": "{:,.2f}"
})

In [ ]:
percentile_levels = [
    0.05,
    0.25,
    0.50,
    0.75,
    0.95
]

murder_rate_percentiles = (
    state["Murder.Rate"]
    .quantile(percentile_levels)
    .rename("Murder Rate")
    .to_frame()
)

murder_rate_percentiles.index = [
    "5%",
    "25%",
    "50%",
    "75%",
    "95%"
]

murder_rate_percentiles

### Analisis Variability

Standard deviation populasi sekitar 6,85 juta, sedangkan IQR sekitar
4,85 juta. Standard deviation menjadi besar karena perhitungannya
dipengaruhi oleh negara bagian dengan populasi sangat tinggi.

IQR dan MAD lebih robust karena tidak terlalu dipengaruhi oleh nilai
ekstrem. Persentil juga membantu memahami letak data tanpa harus
mengasumsikan bentuk distribusi tertentu.

## 4. Exploring the Data Distribution

Distribusi menunjukkan bagaimana nilai-nilai tersebar dalam dataset.

Beberapa visualisasi yang dapat digunakan adalah:

- **Boxplot** untuk melihat median, kuartil, penyebaran, dan outlier.
- **Histogram** untuk melihat jumlah data pada setiap interval.
- **Density plot** untuk memperkirakan bentuk distribusi secara halus.
- **Frequency table** untuk menampilkan jumlah data dalam setiap interval.

In [ ]:
plt.figure(figsize=(6, 5))

sns.boxplot(
    y=state["Population"] / 1_000_000,
    color="steelblue"
)

plt.ylabel("Population (millions)")
plt.title("Boxplot of State Population")
plt.tight_layout()
plt.show()

In [ ]:
population_bins = pd.cut(
    state["Population"],
    bins=10
)

frequency_table = (
    state
    .assign(PopulationBin=population_bins)
    .groupby(
        "PopulationBin",
        observed=False
    )
    .agg(
        Frequency=("State", "size"),
        States=(
            "Abbreviation",
            lambda values: ", ".join(values)
        )
    )
    .reset_index()
)

frequency_table

In [ ]:
figure, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5)
)

sns.histplot(
    state["Population"] / 1_000_000,
    bins=10,
    color="steelblue",
    edgecolor="black",
    ax=axes[0]
)

axes[0].set_xlabel("Population (millions)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Histogram of State Population")

sns.histplot(
    state["Murder.Rate"],
    bins=10,
    stat="density",
    color="lightgray",
    edgecolor="black",
    ax=axes[1]
)

sns.kdeplot(
    state["Murder.Rate"],
    color="darkred",
    linewidth=2,
    ax=axes[1]
)

axes[1].set_xlabel("Murder Rate per 100,000")
axes[1].set_ylabel("Density")
axes[1].set_title("Density of Murder Rate")

plt.tight_layout()
plt.show()

### Analisis Distribusi

Boxplot dan histogram memperlihatkan bahwa sebagian besar negara bagian
memiliki populasi relatif kecil, sedangkan beberapa negara bagian
memiliki populasi jauh lebih tinggi.

Distribusi tersebut tidak simetris dan mempunyai ekor ke kanan.
Kondisi ini menjelaskan mengapa mean populasi lebih besar daripada
median.

Density plot memberikan bentuk distribusi yang lebih halus daripada
histogram. Namun, bentuk density plot dapat berubah tergantung pada
parameter smoothing yang digunakan.

## 5. Exploring Binary and Categorical Data

Data kategorikal berisi nilai yang merepresentasikan kelompok atau
kategori.

Ukuran yang sering digunakan adalah:

- **Mode**, yaitu kategori yang paling sering muncul.
- **Proportion**, yaitu bagian suatu kategori dibandingkan seluruh data.
- **Expected value**, yaitu hasil rata-rata yang diharapkan berdasarkan
  kemungkinan setiap kategori.

Bar chart cocok untuk data kategorikal karena tinggi setiap batang
menunjukkan jumlah atau proporsi suatu kategori.

In [ ]:
delay_counts = dfw_airline.iloc[0]

delay_summary = pd.DataFrame({
    "Count": delay_counts,
    "Percentage": (
        delay_counts
        / delay_counts.sum()
        * 100
    )
}).sort_values(
    "Percentage",
    ascending=False
)

delay_summary.style.format({
    "Count": "{:,.2f}",
    "Percentage": "{:.2f}%"
})

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    x=delay_summary.index,
    y=delay_summary["Percentage"],
    color="steelblue"
)

plt.xlabel("Cause of Delay")
plt.ylabel("Percentage")
plt.title("Causes of Flight Delays")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Analisis Data Kategorikal

Kategori `Inbound` mempunyai persentase terbesar, yaitu sekitar 42,43%.
Kategori berikutnya adalah `ATC` sekitar 30,40% dan `Carrier` sekitar
23,02%.

`Security` memiliki persentase paling kecil, yaitu sekitar 0,12%.
Dengan demikian, penyebab keterlambatan yang paling dominan pada
dataset ini adalah keterlambatan penerbangan masuk.

## 6. Correlation

Korelasi mengukur kekuatan dan arah hubungan linear antara dua variabel
numerik.

Koefisien korelasi Pearson dirumuskan sebagai:

$$
r =
\frac{
\sum (x_i-\bar{x})(y_i-\bar{y})
}{
(n-1)s_xs_y
}
$$

Nilai korelasi berada antara -1 dan 1.

- Mendekati 1: hubungan linear positif yang kuat.
- Mendekati -1: hubungan linear negatif yang kuat.
- Mendekati 0: hubungan linear lemah atau tidak ada.

Korelasi tidak membuktikan hubungan sebab-akibat. Korelasi juga hanya
mengukur hubungan linear sehingga pola non-linear dapat tidak terdeteksi.

## 6. Correlation

Korelasi mengukur kekuatan dan arah hubungan linear antara dua variabel
numerik.

Koefisien korelasi Pearson dirumuskan sebagai:

$$
r =
\frac{
\sum (x_i-\bar{x})(y_i-\bar{y})
}{
(n-1)s_xs_y
}
$$

Nilai korelasi berada antara -1 dan 1.

- Mendekati 1: hubungan linear positif yang kuat.
- Mendekati -1: hubungan linear negatif yang kuat.
- Mendekati 0: hubungan linear lemah atau tidak ada.

Korelasi tidak membuktikan hubungan sebab-akibat. Korelasi juga hanya
mengukur hubungan linear sehingga pola non-linear dapat tidak terdeteksi.

In [ ]:
telecom_symbols = (
    sp500_sectors
    .loc[
        sp500_sectors["sector"]
        == "telecommunications_services",
        "symbol"
    ]
    .tolist()
)

telecom = sp500_data.loc[
    sp500_data.index >= "2012-07-01",
    telecom_symbols
]

telecom_correlation = telecom.corr()

telecom_correlation.round(3)

In [ ]:
etf_symbols = (
    sp500_sectors
    .loc[
        sp500_sectors["sector"] == "etf",
        "symbol"
    ]
    .tolist()
)

etfs = sp500_data.loc[
    sp500_data.index >= "2012-07-01",
    etf_symbols
]

plt.figure(figsize=(11, 8))

sns.heatmap(
    etfs.corr(),
    vmin=-1,
    vmax=1,
    center=0,
    cmap="coolwarm",
    square=True
)

plt.title("Correlation Heatmap of ETF Returns")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(
    telecom["T"],
    telecom["VZ"],
    facecolors="none",
    edgecolors="steelblue",
    alpha=0.6
)

plt.axhline(
    0,
    color="gray",
    linewidth=1
)

plt.axvline(
    0,
    color="gray",
    linewidth=1
)

plt.xlabel("AT&T Daily Return (T)")
plt.ylabel("Verizon Daily Return (VZ)")
plt.title("AT&T versus Verizon Daily Returns")
plt.tight_layout()
plt.show()

print(
    "Correlation between T and VZ:",
    round(
        telecom["T"].corr(
            telecom["VZ"]
        ),
        3
    )
)

### Analisis Korelasi

AT&T dan Verizon mempunyai korelasi positif sekitar 0,678. Ketika
return saham AT&T meningkat, return Verizon cenderung bergerak ke arah
yang sama.

Hubungan tersebut cukup kuat, tetapi tidak sempurna. Selain itu,
korelasi ini tidak berarti bahwa perubahan saham AT&T secara langsung
menyebabkan perubahan saham Verizon.

## 7. Exploring Two or More Variables

Hubungan dua atau lebih variabel dapat dianalisis berdasarkan kombinasi
tipe datanya:

1. Numerik dengan numerik.
2. Kategorikal dengan kategorikal.
3. Kategorikal dengan numerik.
4. Lebih dari dua variabel.

Scatterplot cocok untuk data berukuran kecil. Jika jumlah data sangat
besar, titik-titik dapat saling menutupi. Hexagonal binning dan contour
plot dapat digunakan untuk menampilkan kepadatan data.

In [ ]:
kc_tax_filtered = kc_tax.loc[
    (
        kc_tax["TaxAssessedValue"]
        < 750_000
    )
    & (
        kc_tax["SqFtTotLiving"]
        > 100
    )
    & (
        kc_tax["SqFtTotLiving"]
        < 3_500
    )
].copy()

print(
    "Ukuran data hasil filter:",
    kc_tax_filtered.shape
)

print(
    "Korelasi luas dan nilai properti:",
    round(
        kc_tax_filtered[
            [
                "SqFtTotLiving",
                "TaxAssessedValue"
            ]
        ].corr().iloc[0, 1],
        3
    )
)

In [ ]:
plt.figure(figsize=(8, 6))

hexbin_plot = plt.hexbin(
    kc_tax_filtered["SqFtTotLiving"],
    kc_tax_filtered["TaxAssessedValue"],
    gridsize=30,
    cmap="viridis",
    mincnt=1
)

plt.colorbar(
    hexbin_plot,
    label="Number of Properties"
)

plt.xlabel("Finished Square Feet")
plt.ylabel("Tax Assessed Value")
plt.title("Property Size versus Tax Assessed Value")
plt.tight_layout()
plt.show()

In [ ]:
kc_tax_sample = kc_tax_filtered.sample(
    n=10_000,
    random_state=42
)

plt.figure(figsize=(8, 6))

sns.kdeplot(
    data=kc_tax_sample,
    x="SqFtTotLiving",
    y="TaxAssessedValue",
    fill=True,
    levels=10,
    thresh=0.05,
    cmap="Blues"
)

plt.xlabel("Finished Square Feet")
plt.ylabel("Tax Assessed Value")
plt.title("Density Contour of Property Data")
plt.tight_layout()
plt.show()

### Analisis Numerik dengan Numerik

Luas bangunan dan nilai pajak properti mempunyai korelasi positif
sekitar 0,531. Secara umum, properti yang lebih luas cenderung mempunyai
nilai pajak lebih tinggi.

Hexagonal binning lebih mudah dibaca dibandingkan scatterplot biasa
karena dataset memiliki lebih dari 400.000 baris. Warna yang lebih kuat
menunjukkan wilayah dengan jumlah properti yang lebih banyak.

Hubungan kedua variabel tidak sempurna karena nilai properti juga dapat
dipengaruhi lokasi, kondisi bangunan, tahun pembangunan, dan faktor lain.

In [ ]:
loan_count_table = pd.crosstab(
    lc_loans["grade"],
    lc_loans["status"],
    margins=True
)

loan_percentage_table = (
    pd.crosstab(
        lc_loans["grade"],
        lc_loans["status"],
        normalize="index"
    )
    * 100
)

print("Jumlah pinjaman:")
display(loan_count_table)

print("\nPersentase status pada setiap grade:")
display(
    loan_percentage_table.round(2)
)

### Analisis Kategorikal dengan Kategorikal

Crosstab menunjukkan hubungan antara grade pinjaman dan status
pembayaran.

Persentase `Charged Off` meningkat dari sekitar 2,15% pada grade A
menjadi sekitar 12,62% pada grade G. Persentase `Late` juga meningkat
dari sekitar 0,65% pada grade A menjadi sekitar 6,14% pada grade G.

Hasil ini menunjukkan bahwa grade yang lebih rendah berhubungan dengan
risiko gagal bayar dan keterlambatan yang lebih tinggi.

In [ ]:
figure, axes = plt.subplots(
    1,
    2,
    figsize=(16, 6)
)

sns.boxplot(
    data=airline_stats,
    x="airline",
    y="pct_carrier_delay",
    color="lightblue",
    ax=axes[0]
)

axes[0].set_xlabel("Airline")
axes[0].set_ylabel("Daily Percentage of Carrier Delay")
axes[0].set_title("Boxplot of Carrier Delays")
axes[0].tick_params(
    axis="x",
    rotation=45
)

sns.violinplot(
    data=airline_stats,
    x="airline",
    y="pct_carrier_delay",
    inner="quartile",
    cut=0,
    color="lightgreen",
    ax=axes[1]
)

axes[1].set_xlabel("Airline")
axes[1].set_ylabel("Daily Percentage of Carrier Delay")
axes[1].set_title("Violin Plot of Carrier Delays")
axes[1].tick_params(
    axis="x",
    rotation=45
)

plt.tight_layout()
plt.show()

### Analisis Kategorikal dengan Numerik

Boxplot membandingkan median, kuartil, penyebaran, dan outlier
persentase keterlambatan setiap maskapai.

American memiliki median keterlambatan sekitar 8,43%, sedangkan Alaska
memiliki median sekitar 3,23%. Pada dataset ini, Alaska mempunyai median
keterlambatan akibat maskapai yang paling rendah.

Violin plot memberikan informasi tambahan mengenai bentuk dan
kepadatan distribusi. Bagian violin yang lebih lebar menunjukkan lebih
banyak data berada pada rentang tersebut.

In [ ]:
selected_zip_codes = [
    98188,
    98105,
    98108,
    98126
]

kc_tax_by_zip = kc_tax_filtered.loc[
    kc_tax_filtered["ZipCode"].isin(
        selected_zip_codes
    )
].copy()

kc_tax_by_zip["ZipCode"] = (
    kc_tax_by_zip["ZipCode"]
    .astype(int)
    .astype(str)
)

def draw_hexbin(data, **kwargs):
    plt.hexbin(
        data["SqFtTotLiving"],
        data["TaxAssessedValue"],
        gridsize=25,
        cmap="Blues",
        mincnt=1
    )

facet = sns.FacetGrid(
    kc_tax_by_zip,
    col="ZipCode",
    col_wrap=2,
    height=4,
    sharex=True,
    sharey=True
)

facet.map_dataframe(
    draw_hexbin
)

facet.set_axis_labels(
    "Finished Square Feet",
    "Tax Assessed Value"
)

facet.set_titles(
    "ZIP Code {col_name}"
)

facet.figure.suptitle(
    "Property Value by Size and ZIP Code",
    y=1.03
)

plt.show()

### Analisis Multivariat

Visualisasi ini menghubungkan tiga variabel, yaitu luas bangunan,
nilai pajak properti, dan ZIP Code.

Hubungan antara luas dan nilai properti tidak sepenuhnya sama pada
setiap ZIP Code. Hal ini menunjukkan bahwa lokasi dapat memengaruhi
nilai properti.

Jika seluruh ZIP Code digabungkan, perbedaan antarlokasi tersebut dapat
tersembunyi. Facet plot membantu melihat pola pada setiap kelompok
secara terpisah.

## 10. References

1. Bruce, P., Bruce, A., & Gedeck, P. (2020).
   *Practical Statistics for Data Scientists: 50+ Essential Concepts
   Using R and Python* (2nd ed.). O'Reilly Media.

2. Official Book Repository.  
   https://github.com/gedeck/practical-statistics-for-data-scientists

3. Pandas Documentation.  
   https://pandas.pydata.org/docs/

4. SciPy Documentation.  
   https://docs.scipy.org/doc/scipy/

5. Matplotlib Documentation.  
   https://matplotlib.org/stable/

6. Seaborn Documentation.  
   https://seaborn.pydata.org/